# 🎙️ Транскрибатор видео/аудио с разделением спикеров

**Стек:** faster-whisper (large-v3-turbo) + pyannote.audio 3.x + ffmpeg


## 2. Проверка GPU

In [1]:
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
print(f"CUDA версия: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ CUDA не найдена! Проверь установку PyTorch и драйверов. Процесс будем проходить на CPU")

PyTorch: 2.11.0+cu128
CUDA доступна: True
CUDA версия: 12.8
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 15.9 GB


## 3. Настройки и токен

In [ ]:
# ПАРАМЕТРЫ

# Папки
INPUT_DIR = "media"     # папка с исходными файлами
OUTPUT_DIR = "output"   # папка для результатов !!!

# Создаём папки, если их нет
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Путь к файлу (видео или аудио), поддержка: mp4, mkv, avi, mov, mp3, wav, flac, ogg...
# Имя файла (без пути — просто имя внутри media/)
FILENAME = "962b355e-7e1d-4c44-9e00-5a2b9489c3d2.mp3"

# Полные пути
INPUT_FILE = os.path.join(INPUT_DIR, FILENAME)
base_name = os.path.splitext(FILENAME)[0]
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"{base_name}_transcript.txt")



# ТОКЕН HuggingFace  (он бесплатный)
HF_TOKEN = " ВСТАВЬ СЮДА СВОЙ ТОКЕН"




# Язык (None = автоопределение, 'ru', 'en', 'de' и т.д.) даже если выбрать ru, то транскрибатор может писать анлийские слова
LANGUAGE = None

# Количество спикеров (None = авто), тут надо тестить
NUM_SPEAKERS = None

# Модель Whisper (можно и другую версию)
WHISPER_MODEL = "large-v3-turbo"

print(f"Вход:  {INPUT_FILE}")
print(f"Выход: {OUTPUT_FILE}")

Вход:  media\962b355e-7e1d-4c44-9e00-5a2b9489c3d2.mp3
Выход: output\962b355e-7e1d-4c44-9e00-5a2b9489c3d2_transcript.txt


## 4. Извлечение аудио из видео или преобразование аудио

**Обязательно** нужно изменить формат, так как pyannote будет ругаться на чанки.



In [ ]:
import subprocess


AUDIO_FILE = os.path.join(OUTPUT_DIR, "audio_extracted.wav")

audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".wma"}
ext = os.path.splitext(INPUT_FILE)[1].lower()

if ext in audio_extensions and ext == ".wav":
    # Уже WAV - просто конвертируем в нужный формат
    print("Входной файл - WAV, конвертируем в 16kHz mono...")
    subprocess.run([
        "ffmpeg", "-y", "-i", INPUT_FILE,
        "-ar", "16000", "-ac", "1", AUDIO_FILE
    ], check=True)
else:
    print(f"Извлекаем аудио из {INPUT_FILE}...")
    subprocess.run([
        "ffmpeg", "-y", "-i", INPUT_FILE,
        "-vn",           # без видео
        "-acodec", "pcm_s16le",  # PCM 16-bit
        "-ar", "16000",  # 16 kHz
        "-ac", "1",      # моно
        AUDIO_FILE
    ], check=True)

size_mb = os.path.getsize(AUDIO_FILE) / 1024**2
print(f"✅ Аудио готово: {AUDIO_FILE} ({size_mb:.1f} MB)")

Извлекаем аудио из media\962b355e-7e1d-4c44-9e00-5a2b9489c3d2.mp3...
✅ Аудио готово: output\audio_extracted.wav (97.5 MB)


## 5. Диаризация (pyannote)

In [4]:
from pyannote.audio import Pipeline
import time

print("Загрузка модели диаризации pyannote...")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN # наш токен
)
diarization_pipeline.to(torch.device("cuda"))

print("Запуск диаризацию...")
t0 = time.time()

diarization_params = {}
if NUM_SPEAKERS is not None:
    diarization_params["num_speakers"] = NUM_SPEAKERS

output = diarization_pipeline(AUDIO_FILE, **diarization_params) # 

if hasattr(output, "speaker_diarization"):
    diarization = output.speaker_diarization
else:
    diarization = output

elapsed = time.time() - t0
print(f"Диаризация завершена за {elapsed:.1f} сек")

# Найденные собеседники
speakers = set()
for turn, _, speaker in diarization.itertracks(yield_label=True):
    speakers.add(speaker)
print(f"Найдено говорящих: {len(speakers)} — {sorted(speakers)}")


c:\Users\dokto\anaconda3\envs\for_whisper\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0413 21:39:15.864000 13792 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Загрузка модели диаризации pyannote...
Запуск диаризацию...


c:\Users\dokto\anaconda3\envs\for_whisper\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
c:\Users\dokto\anaconda3\envs\for_whisper\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


Диаризация завершена за 127.9 сек
Найдено говорящих: 5 — ['SPEAKER_00', 'SPEAKER_01', 'SPEAKER_02', 'SPEAKER_03', 'SPEAKER_04']


## 6. Транскрибация

In [5]:
from faster_whisper import WhisperModel

print(f"Загружаем модель {WHISPER_MODEL}...")
whisper_model = WhisperModel(
    WHISPER_MODEL,
    device="cuda",
    compute_type="float16"  # float16
)

print("Транскрибируем всё аудио...")
t0 = time.time()

segments_raw, info = whisper_model.transcribe(
    AUDIO_FILE,
    language=LANGUAGE,
    beam_size=5,
    vad_filter=True,           # фильтр тишины
    vad_parameters=dict(
        min_silence_duration_ms=500
    ),
    word_timestamps=True       # точные таймкоды слов
)

whisper_segments = []
for seg in segments_raw:
    whisper_segments.append({
        "start": seg.start,
        "end": seg.end,
        "text": seg.text.strip(),
        "words": seg.words
    })

elapsed = time.time() - t0
detected_lang = info.language
print(f"✅ Транскрипция завершена за {elapsed:.1f} сек")
print(f"Язык: {detected_lang} (вероятность: {info.language_probability:.2%})")
print(f"Сегментов: {len(whisper_segments)}")

Загружаем модель large-v3-turbo...
Транскрибируем всё аудио...
✅ Транскрипция завершена за 93.4 сек
Язык: ru (вероятность: 99.95%)
Сегментов: 1404


## 7. Соединяем все: спикер + текст + таймкод

In [6]:
def format_time(seconds):
    """Секунды -> HH:MM:SS"""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def assign_speaker(seg_start, seg_end, diarization):
    """Определяем спикера для сегмента по максимальному пересечению."""
    speaker_durations = {}
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        overlap_start = max(seg_start, turn.start)
        overlap_end = min(seg_end, turn.end)
        overlap = max(0, overlap_end - overlap_start)
        if overlap > 0:
            speaker_durations[speaker] = speaker_durations.get(speaker, 0) + overlap
    if not speaker_durations:
        return "UNKNOWN"
    return max(speaker_durations, key=speaker_durations.get)


# Склеиваем
transcript_lines = []
prev_speaker = None

for seg in whisper_segments:
    speaker = assign_speaker(seg["start"], seg["end"], diarization)
    time_str = format_time(seg["start"])
    
    # Группируем последовательные фразы одного спикера
    if speaker == prev_speaker and transcript_lines:
        transcript_lines[-1]["text"] += " " + seg["text"]
        transcript_lines[-1]["end"] = seg["end"]
    else:
        transcript_lines.append({
            "speaker": speaker,
            "start": seg["start"],
            "end": seg["end"],
            "text": seg["text"]
        })
    prev_speaker = speaker

print(f"✅ Итого реплик: {len(transcript_lines)}")

✅ Итого реплик: 169


## 8. Результат

In [ ]:
# Красивый вывод в ноутбуке (эту создал ИИ)
from IPython.display import display, HTML

# Цвета для каждого спикера
speaker_colors = {}
palette = ["#4A90D9", "#E74C3C", "#2ECC71", "#F39C12", "#9B59B6", "#1ABC9C", "#E67E22", "#3498DB"]

html_parts = ['<div style="font-family: monospace; line-height: 1.8;">']
for line in transcript_lines:
    sp = line["speaker"]
    if sp not in speaker_colors:
        speaker_colors[sp] = palette[len(speaker_colors) % len(palette)]
    color = speaker_colors[sp]
    t_start = format_time(line["start"])
    t_end = format_time(line["end"])
    html_parts.append(
        f'<p><span style="color:{color}; font-weight:bold;">[{t_start} → {t_end}] {sp}:</span> '
        f'{line["text"]}</p>'
    )
html_parts.append('</div>')

# display - можно вывести на экран, можно и убрать
display(HTML("".join(html_parts)))

In [8]:
# Сохранение цветного транскрипта в HTML-файл
html_output_file = os.path.splitext(OUTPUT_FILE)[0] + ".html"

lines_html = []
for line in transcript_lines:
    color = speaker_colors.get(line["speaker"], "#fff")
    t_start = format_time(line["start"])
    t_end = format_time(line["end"])
    lines_html.append(
        f'<div class="line">'
        f'<span class="speaker" style="color:{color};">'
        f'[{t_start} \u2192 {t_end}] {line["speaker"]}:'
        f'</span> {line["text"]}</div>'
    )

full_html = "\n".join([
    "<!DOCTYPE html>",
    '<html lang="ru">',
    "<head>",
    '  <meta charset="UTF-8">',
    "  <title>Транскрипт</title>",
    "  <style>",
    "    body { font-family: monospace; font-size: 14px; line-height: 1.8;",
    "           max-width: 900px; margin: 40px auto; padding: 0 20px;",
    "           background: #1e1e1e; color: #d4d4d4; }",
    "    .line { margin-bottom: 6px; }",
    "    .speaker { font-weight: bold; }",
    "  </style>",
    "</head>",
    "<body>",
    *lines_html,
    "</body>",
    "</html>",
])

with open(html_output_file, "w", encoding="utf-8") as f:
    f.write(full_html)

print(f"✅ HTML транскрипт сохранён в {html_output_file}")

✅ HTML транскрипт сохранён в output\962b355e-7e1d-4c44-9e00-5a2b9489c3d2_transcript.html


## 9. Сохранение в файл

In [9]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for line in transcript_lines:
        t_start = format_time(line["start"])
        t_end = format_time(line["end"])
        f.write(f"[{t_start} → {t_end}] {line['speaker']}:\n")
        f.write(f"{line['text']}\n\n")

print(f"✅ Сохранено в {OUTPUT_FILE}")

✅ Сохранено в output\962b355e-7e1d-4c44-9e00-5a2b9489c3d2_transcript.txt


## 10. Экспорт в SRT (субтитры)

In [10]:
def format_srt_time(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int((seconds % 1) * 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

srt_file = os.path.join(OUTPUT_DIR, f"{base_name}_transcript.srt")
with open(srt_file, "w", encoding="utf-8") as f:
    for i, line in enumerate(transcript_lines, 1):
        f.write(f"{i}\n")
        f.write(f"{format_srt_time(line['start'])} --> {format_srt_time(line['end'])}\n")
        f.write(f"[{line['speaker']}] {line['text']}\n\n")

print(f"✅ Субтитры сохранены в {srt_file}")

✅ Субтитры сохранены в output\962b355e-7e1d-4c44-9e00-5a2b9489c3d2_transcript.srt


## 11. Очистка временных файлов

In [11]:
if os.path.exists(AUDIO_FILE):
    os.remove(AUDIO_FILE)
    print(f"Удалён: {AUDIO_FILE}")
print("Готово!")

Удалён: output\audio_extracted.wav
Готово!
